In [1]:
import warnings
import pandas as pd
import numpy as np

# Silence harmless binning warnings on skewed features
warnings.filterwarnings("ignore", category=UserWarning)

# Load shared datasets
train = pd.read_csv("shared_data/train_clean.csv")
val = pd.read_csv("shared_data/val_clean.csv")
test = pd.read_csv("shared_data/test_clean.csv")

target = "is_canceled"

X_train = train.drop(columns=[target])
y_train = train[target]

X_val = val.drop(columns=[target])
y_val = val[target]

X_test = test.drop(columns=[target])
y_test = test[target]

# Check shapes
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# Check columns match
assert X_train.columns.equals(X_val.columns)
assert X_train.columns.equals(X_test.columns)

# Check no missing values
assert X_train.isna().sum().sum() == 0
assert X_val.isna().sum().sum() == 0
assert X_test.isna().sum().sum() == 0

print("Data validation: PASSED")

Train: (83815, 24)
Validation: (18237, 24)
Test: (17338, 24)
Data validation: PASSED


In [2]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# Always predict the majority class
baseline = DummyClassifier(
    strategy="most_frequent"
)

baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)

print("Baseline Accuracy:",
      accuracy_score(y_val, baseline_pred))

print("Baseline Precision:",
      precision_score(y_val, baseline_pred, zero_division=0))

print("Baseline Recall:",
      recall_score(y_val, baseline_pred, zero_division=0))

print("Baseline F1:",
      f1_score(y_val, baseline_pred, zero_division=0))

Baseline Accuracy: 0.6111202500411251
Baseline Precision: 0.0
Baseline Recall: 0.0
Baseline F1: 0.0


In [3]:
categorical_features = [
    "hotel",
    "arrival_date_month",
    "meal",
    "country",
    "market_segment",
    "distribution_channel",
    "reserved_room_type",
    "deposit_type",
    "customer_type"
]

numerical_features = [
    col for col in X_train.columns
    if col not in categorical_features
]

print("Categorical:", len(categorical_features))
print("Numerical:", len(numerical_features))

# Validation protocol (same as logistic_regression.ipynb):
#  1. hyperparameters: fit on train, keep the value with the best validation F1 (default 0.5 cut-off)
#  2. threshold: fixed grid below, keep the value with the best validation F1
#  3. test set: used once, with the chosen hyperparameters and threshold
THRESHOLDS = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

Categorical: 9
Numerical: 15


In [4]:
# GaussianNB: one-hot categoricals + standardised numerics
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB

gnb_preprocessor = ColumnTransformer([
    ("num", StandardScaler(), numerical_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
     categorical_features)
])

gnb_pipeline = Pipeline([
    ("preprocessor", gnb_preprocessor),
    ("classifier", GaussianNB())
])

gnb_pipeline.fit(X_train, y_train)

from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred = gnb_pipeline.predict(X_val)

print(classification_report(
    y_val, y_pred,
    target_names=["Not Canceled", "Canceled"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("ROC-AUC:", roc_auc_score(y_val, gnb_pipeline.predict_proba(X_val)[:, 1]))

              precision    recall  f1-score   support

Not Canceled       0.83      0.06      0.12     11145
    Canceled       0.40      0.98      0.57      7092

    accuracy                           0.42     18237
   macro avg       0.61      0.52      0.34     18237
weighted avg       0.66      0.42      0.29     18237

Confusion Matrix:
[[  717 10428]
 [  150  6942]]
ROC-AUC: 0.5240739222528649


In [5]:
# Tune GaussianNB var_smoothing on validation F1 (best vs best comparison)
from sklearn.base import clone

gnb_results = []

for vs in [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]:
    model = clone(gnb_pipeline)
    model.set_params(classifier__var_smoothing=vs)

    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    prob = model.predict_proba(X_val)[:, 1]

    gnb_results.append({
        "var_smoothing": vs,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred),
        "ROC-AUC": roc_auc_score(y_val, prob)
    })

gnb_tuning_results = pd.DataFrame(gnb_results)
print(gnb_tuning_results.to_string(float_format=lambda v: f"{v:.4g}"))
print("\nBest GaussianNB val F1:", round(gnb_tuning_results["F1"].max(), 4))


   var_smoothing  Accuracy  Precision  Recall     F1  ROC-AUC
0          1e-09      0.42     0.3997  0.9788 0.5676   0.5241
1          1e-07    0.4304     0.4039  0.9759 0.5713   0.5338
2          1e-05    0.4984     0.4319  0.9188 0.5876   0.7614
3          0.001    0.6544      0.538  0.7869 0.6391   0.7883
4            0.1    0.6233     0.5091  0.8768 0.6442   0.8094

Best GaussianNB val F1: 0.6442


In [6]:
# CategoricalNB: numerics binned into quantile bins; unseen categories -> index 0
from sklearn.preprocessing import OrdinalEncoder, KBinsDiscretizer, FunctionTransformer
from sklearn.naive_bayes import CategoricalNB

cnb_preprocessor = ColumnTransformer([
    ("num", KBinsDiscretizer(
        n_bins=10,
        encode="ordinal",
        strategy="quantile",
        quantile_method="averaged_inverted_cdf"
    ), numerical_features),
    ("cat", Pipeline([
        ("ordinal", OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1
        )),
        ("shift", FunctionTransformer(lambda a: a + 1))
    ]), categorical_features)
])

cnb_pipeline = Pipeline([
    ("preprocessor", cnb_preprocessor),
    ("classifier", CategoricalNB(alpha=1.0))
])

cnb_pipeline.fit(X_train, y_train)

y_pred = cnb_pipeline.predict(X_val)

print(classification_report(
    y_val, y_pred,
    target_names=["Not Canceled", "Canceled"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))

print("ROC-AUC:", roc_auc_score(y_val, cnb_pipeline.predict_proba(X_val)[:, 1]))

              precision    recall  f1-score   support

Not Canceled       0.76      0.90      0.83     11145
    Canceled       0.78      0.55      0.65      7092

    accuracy                           0.77     18237
   macro avg       0.77      0.73      0.74     18237
weighted avg       0.77      0.77      0.76     18237

Confusion Matrix:
[[10072  1073]
 [ 3190  3902]]
ROC-AUC: 0.8218535686966933


In [7]:
# Tune alpha and n_bins on validation F1
from sklearn.base import clone

results = []

for n_bins in [5, 10, 20]:
    for alpha in [0.01, 0.1, 1, 10]:
        model = clone(cnb_pipeline)
        model.set_params(
            preprocessor__num__n_bins=n_bins,
            classifier__alpha=alpha
        )

        model.fit(X_train, y_train)
        pred = model.predict(X_val)
        prob = model.predict_proba(X_val)[:, 1]

        results.append({
            "n_bins": n_bins,
            "alpha": alpha,
            "Accuracy": accuracy_score(y_val, pred),
            "Precision": precision_score(y_val, pred),
            "Recall": recall_score(y_val, pred),
            "F1": f1_score(y_val, pred),
            "ROC-AUC": roc_auc_score(y_val, prob)
        })

tuning_results = pd.DataFrame(results)
print(tuning_results.round(4))

    n_bins  alpha  Accuracy  Precision  Recall      F1  ROC-AUC
0        5   0.01    0.7652     0.7791  0.5530  0.6469   0.8217
1        5   0.10    0.7653     0.7793  0.5530  0.6469   0.8217
2        5   1.00    0.7650     0.7792  0.5523  0.6464   0.8216
3        5  10.00    0.7668     0.7845  0.5519  0.6480   0.8209
4       10   0.01    0.7662     0.7837  0.5508  0.6469   0.8220
5       10   0.10    0.7662     0.7837  0.5508  0.6469   0.8220
6       10   1.00    0.7662     0.7843  0.5502  0.6467   0.8219
7       10  10.00    0.7655     0.7852  0.5464  0.6444   0.8213
8       20   0.01    0.7707     0.8038  0.5429  0.6480   0.8270
9       20   0.10    0.7706     0.8037  0.5426  0.6478   0.8270
10      20   1.00    0.7706     0.8042  0.5420  0.6476   0.8269
11      20  10.00    0.7707     0.8073  0.5392  0.6465   0.8264


In [8]:
# Pick best setting by validation F1
best = tuning_results.sort_values("F1", ascending=False).iloc[0]
print("Best setting:")
print(best)

model = clone(cnb_pipeline)
model.set_params(
    preprocessor__num__n_bins=int(best["n_bins"]),
    classifier__alpha=best["alpha"]
)
model.fit(X_train, y_train)

val_prob = model.predict_proba(X_val)[:, 1]

thresholds = THRESHOLDS
results = []

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)

    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_val, pred),
        "Precision": precision_score(y_val, pred),
        "Recall": recall_score(y_val, pred),
        "F1": f1_score(y_val, pred)
    })

threshold_results = pd.DataFrame(results)
print(threshold_results.round(4))
print("\nBest val F1 -> GaussianNB:", round(gnb_tuning_results["F1"].max(), 4),
      "| CategoricalNB:", round(tuning_results["F1"].max(), 4))

Best setting:
n_bins       20.000000
alpha         0.010000
Accuracy      0.770686
Precision     0.803758
Recall        0.542865
F1            0.648039
ROC-AUC       0.826990
Name: 8, dtype: float64
   Threshold  Accuracy  Precision  Recall      F1
0       0.30    0.7373     0.6478  0.7112  0.6780
1       0.35    0.7524     0.6874  0.6661  0.6766
2       0.40    0.7645     0.7298  0.6265  0.6742
3       0.45    0.7692     0.7717  0.5773  0.6605
4       0.50    0.7707     0.8038  0.5429  0.6480
5       0.55    0.7696     0.8328  0.5100  0.6326
6       0.60    0.7711     0.8743  0.4805  0.6202

Best val F1 -> GaussianNB: 0.6442 | CategoricalNB: 0.648


In [9]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["Recall"],
    marker="s",
    label="Recall"
)

plt.plot(
    threshold_results["Threshold"],
    threshold_results["F1"],
    marker="^",
    label="F1"
)

plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title("Naive Bayes Validation Performance by Threshold")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [10]:
# Pick threshold by validation F1
final_threshold = threshold_results.loc[
    threshold_results["F1"].idxmax(), "Threshold"
]
print("Selected threshold:", final_threshold)

# Predict on the test set
test_prob = model.predict_proba(X_test)[:, 1]

# Apply the threshold
test_pred = (test_prob >= final_threshold).astype(int)

# Final evaluation
print("Accuracy:", accuracy_score(y_test, test_pred))
print("Precision:", precision_score(y_test, test_pred))
print("Recall:", recall_score(y_test, test_pred))
print("F1-score:", f1_score(y_test, test_pred))
print("ROC-AUC:", roc_auc_score(y_test, test_prob))

print("\nClassification Report:")
print(classification_report(y_test, test_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_pred))

Selected threshold: 0.3
Accuracy: 0.7402814626831238
Precision: 0.6221532091097308
Recall: 0.6835093419983753
F1-score: 0.6513896415576372
ROC-AUC: 0.8145966813239864

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.77      0.79     11183
           1       0.62      0.68      0.65      6155

    accuracy                           0.74     17338
   macro avg       0.72      0.73      0.72     17338
weighted avg       0.75      0.74      0.74     17338


Confusion Matrix:
[[8628 2555]
 [1948 4207]]


In [11]:
# Save the NB tuning / final-test results (same folder as the LR outputs)
from pathlib import Path
from sklearn.metrics import ConfusionMatrixDisplay

output_dir = Path("outputs")
output_dir.mkdir(parents=True, exist_ok=True)

gnb_tuning_results.to_csv(output_dir / "naive_bayes_GaussianNB_tuning.csv", index=False)
tuning_results.to_csv(output_dir / "naive_bayes_CategoricalNB_tuning.csv", index=False)
threshold_results.to_csv(output_dir / "naive_bayes_threshold_tuning.csv", index=False)

pd.DataFrame([{
    "Model": "Naive Bayes (CategoricalNB)",
    "n_bins": int(best["n_bins"]),
    "alpha": best["alpha"],
    "Threshold": final_threshold,
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred),
    "Recall": recall_score(y_test, test_pred),
    "F1": f1_score(y_test, test_pred),
    "ROC_AUC": roc_auc_score(y_test, test_prob),
}]).to_csv(output_dir / "naive_bayes_final_results.csv", index=False)

# Confusion matrix on the test set
ConfusionMatrixDisplay.from_predictions(
    y_test, test_pred,
    display_labels=["Not Canceled", "Canceled"], cmap="Blues"
)
plt.title(f"Naive Bayes Test Confusion Matrix (threshold = {final_threshold})")
plt.tight_layout()
plt.savefig(output_dir / "naive_bayes_confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [12]:
# ============================================================
# ERROR ANALYSIS (test set, final CategoricalNB). Post-hoc only: no model choice depends on it.
# ============================================================
error_df = X_test.copy()
error_df["actual"] = y_test.values
error_df["predicted"] = test_pred
error_df["cancel_probability"] = test_prob
error_df["error_type"] = np.select(
    [(error_df["actual"] == 0) & (error_df["predicted"] == 0),
     (error_df["actual"] == 0) & (error_df["predicted"] == 1),
     (error_df["actual"] == 1) & (error_df["predicted"] == 0)],
    ["TN", "FP", "FN"], default="TP"
)
print(error_df["error_type"].value_counts())

# Mean of key booking features per outcome (same features as the LR error analysis)
numeric_error_features = ["lead_time", "total_nights", "total_guests",
                          "previous_cancellations", "previous_bookings_not_canceled"]
nb_error_summary = error_df.groupby("error_type")[numeric_error_features].mean().round(2)
print("\n", nb_error_summary)

# Error rates by hotel and by deposit type
def error_rates(group_col):
    return error_df.groupby(group_col).apply(
        lambda x: pd.Series({
            "Count": len(x),
            "Cancel_rate": x["actual"].mean(),
            "FP_rate": (x["error_type"] == "FP").mean(),
            "FN_rate": (x["error_type"] == "FN").mean(),
            "Error_rate": (x["actual"] != x["predicted"]).mean(),
        }), include_groups=False)

print("\nBy hotel:\n", error_rates("hotel").round(4))
print("\nBy deposit type:\n", error_rates("deposit_type").round(4))
print("\nBy market segment:\n", error_rates("market_segment").round(4))

nb_error_summary.to_csv(output_dir / "naive_bayes_error_summary.csv")
error_rates("hotel").to_csv(output_dir / "naive_bayes_hotel_error_rate.csv")
error_rates("deposit_type").to_csv(output_dir / "naive_bayes_deposit_error_rate.csv")

error_type
TN    8628
TP    4207
FP    2555
FN    1948
Name: count, dtype: int64

             lead_time  ...  previous_bookings_not_canceled
error_type             ...                                
FN              72.31  ...                            0.03
FP             131.34  ...                            0.19
TN              60.55  ...                            0.19
TP             171.45  ...                            0.02

[4 rows x 5 columns]

By hotel:
                 Count  Cancel_rate  FP_rate  FN_rate  Error_rate
hotel                                                           
City Hotel    11342.0       0.3942   0.1837   0.1095      0.2932
Resort Hotel   5996.0       0.2809   0.0786   0.1177      0.1963

By deposit type:
                 Count  Cancel_rate  FP_rate  FN_rate  Error_rate
deposit_type                                                    
No Deposit    15326.0       0.2721   0.1659    0.127      0.2929
Non Refund     1992.0       0.9940   0.0060    0.000   

In [13]:
# Most confident mistakes (specific cases to discuss in the report)
columns_to_show = ["hotel", "lead_time", "total_nights", "total_guests", "deposit_type",
                   "market_segment", "customer_type", "previous_cancellations",
                   "previous_bookings_not_canceled", "cancel_probability", "actual", "predicted"]

fp_examples = error_df[error_df["error_type"] == "FP"].sort_values("cancel_probability", ascending=False)
fn_examples = error_df[error_df["error_type"] == "FN"].sort_values("cancel_probability", ascending=True)

print("=== Most confident false positives ===")
print(fp_examples[columns_to_show].head(5).to_string(index=False))
print("\n=== Most confident false negatives ===")
print(fn_examples[columns_to_show].head(5).to_string(index=False))

fp_examples[columns_to_show].head(20).to_csv(output_dir / "naive_bayes_FP_examples.csv", index=False)
fn_examples[columns_to_show].head(20).to_csv(output_dir / "naive_bayes_FN_examples.csv", index=False)

=== Most confident false positives ===
       hotel  lead_time  total_nights  total_guests deposit_type market_segment   customer_type  previous_cancellations  previous_bookings_not_canceled  cancel_probability  actual  predicted
  City Hotel        172             4           3.0   No Deposit      Online TA       Transient                       0                               0            0.991124       0          1
Resort Hotel          1             4           2.0   No Deposit      Online TA       Transient                       0                               0            0.987507       0          1
  City Hotel         73             3           1.0   Non Refund      Corporate Transient-Party                       0                              13            0.979031       0          1
  City Hotel        172             3           1.0   Non Refund      Corporate Transient-Party                       0                               0            0.973330       0          1
Resort

In [14]:
# NB vs LR on the SAME test rows: where does NB fail that LR gets right?
# (reads the LR error table saved by logistic_regression.ipynb; skipped if it is missing)
lr_path = output_dir / "logistic_regression_error_analysis.csv"
if lr_path.exists():
    lr_err = pd.read_csv(lr_path)
    assert len(lr_err) == len(error_df) and (lr_err["actual"].values == error_df["actual"].values).all()

    nb_wrong = (error_df["actual"] != error_df["predicted"]).values
    lr_wrong = (lr_err["actual"] != lr_err["predicted"]).values
    print(pd.crosstab(pd.Series(nb_wrong, name="NB wrong"), pd.Series(lr_wrong, name="LR wrong")))

    nb_only = error_df[nb_wrong & ~lr_wrong]
    print("\nNB-only mistakes by type:\n", nb_only["error_type"].value_counts())
    print("\nMean features: NB-only mistakes vs all test rows")
    print(pd.DataFrame({"NB-only wrong": nb_only[numeric_error_features].mean(),
                        "All test": error_df[numeric_error_features].mean()}).round(2))
    print("\nNB-only FP: market segment (share of NB-only FP vs share of all non-cancelled rows)")
    nb_only_fp = nb_only[nb_only["error_type"] == "FP"]
    print(pd.DataFrame({
        "NB-only FP": nb_only_fp["market_segment"].value_counts(normalize=True),
        "All not-cancelled": error_df[error_df["actual"] == 0]["market_segment"].value_counts(normalize=True),
    }).round(3))
else:
    print("LR error table not found -> run logistic_regression.ipynb first")

LR wrong  False  True 
NB wrong              
False     11081   1754
True       1967   2536

NB-only mistakes by type:
 error_type
FP    1102
FN     865
Name: count, dtype: int64

Mean features: NB-only mistakes vs all test rows
                                NB-only wrong  All test
lead_time                              105.48     99.21
total_nights                             3.75      3.40
total_guests                             2.15      1.97
previous_cancellations                   0.03      0.09
previous_bookings_not_canceled           0.22      0.13

NB-only FP: market segment (share of NB-only FP vs share of all non-cancelled rows)
                NB-only FP  All not-cancelled
market_segment                               
Aviation               NaN              0.002
Complementary        0.001              0.008
Corporate            0.025              0.061
Direct               0.004              0.144
Groups               0.302              0.092
Offline TA/TO        0.377  

In [15]:
# Reliability (calibration) curve: are the NB probabilities trustworthy?
# NB multiplies per-feature likelihoods as if features were independent; correlated features
# (total_nights = weekend + week nights, total_guests, month/week/day) are counted repeatedly.
from sklearn.calibration import calibration_curve
from sklearn.metrics import brier_score_loss

plt.figure(figsize=(6, 5))
plt.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")

frac_pos, mean_pred = calibration_curve(y_test, test_prob, n_bins=10, strategy="uniform")
plt.plot(mean_pred, frac_pos, marker="o", label="Naive Bayes")
print("NB   Brier score:", round(brier_score_loss(y_test, test_prob), 4))
print("NB   share of predicted probabilities < 0.1 or > 0.9:",
      round(((test_prob < 0.1) | (test_prob > 0.9)).mean(), 3))

if lr_path.exists():
    lr_prob = lr_err["cancel_probability"].values
    frac_pos, mean_pred = calibration_curve(y_test, lr_prob, n_bins=10, strategy="uniform")
    plt.plot(mean_pred, frac_pos, marker="s", label="Logistic Regression")
    print("LR   Brier score:", round(brier_score_loss(y_test, lr_prob), 4))
    print("LR   share of predicted probabilities < 0.1 or > 0.9:",
          round(((lr_prob < 0.1) | (lr_prob > 0.9)).mean(), 3))

plt.xlabel("Mean predicted probability of cancellation")
plt.ylabel("Observed cancellation rate")
plt.title("Reliability Curve on the Test Set")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(output_dir / "naive_bayes_calibration.png", dpi=300, bbox_inches="tight")
plt.show()

NB   Brier score: 0.1561
NB   share of predicted probabilities < 0.1 or > 0.9: 0.445
LR   Brier score: 0.138
LR   share of predicted probabilities < 0.1 or > 0.9: 0.317


In [16]:
# RQ1: does customer history improve cancellation prediction?
# A: all 24 features; B: drop previous_cancellations, previous_bookings_not_canceled, is_repeated_guest

# Full NB procedure for a given feature set
from sklearn.metrics import roc_auc_score

history_features = [
    "previous_cancellations",
    "previous_bookings_not_canceled",
    "is_repeated_guest"
]

def run_nb_experiment(drop_cols, variant=None):
    """Tune on validation F1, pick the threshold on validation F1, evaluate once on test.
    variant=None keeps the better NB variant by validation F1; pass a name to fix it.
    Returns (metrics dict, test-set probabilities)."""
    Xtr = X_train.drop(columns=drop_cols)
    Xva = X_val.drop(columns=drop_cols)
    Xte = X_test.drop(columns=drop_cols)

    cat = [c for c in categorical_features if c in Xtr.columns]
    num = [c for c in Xtr.columns if c not in cat]

    # GaussianNB: tune var_smoothing
    gnb = Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", StandardScaler(), num),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)
        ])),
        ("classifier", GaussianNB())
    ])
    gnb_scores = {}
    for vs in [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]:
        m = clone(gnb).set_params(classifier__var_smoothing=vs).fit(Xtr, y_train)
        gnb_scores[vs] = f1_score(y_val, m.predict(Xva))
    best_vs = max(gnb_scores, key=gnb_scores.get)

    # CategoricalNB: tune n_bins and alpha
    cnb = Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile",
                                     quantile_method="averaged_inverted_cdf"), num),
            ("cat", Pipeline([
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value",
                                           unknown_value=-1)),
                ("shift", FunctionTransformer(lambda a: a + 1))
            ]), cat)
        ])),
        ("classifier", CategoricalNB())
    ])
    cnb_scores = {}
    for n_bins in [5, 10, 20]:
        for alpha in [0.01, 0.1, 1, 10]:
            m = clone(cnb).set_params(preprocessor__num__n_bins=n_bins,
                                      classifier__alpha=alpha).fit(Xtr, y_train)
            cnb_scores[(n_bins, alpha)] = f1_score(y_val, m.predict(Xva))
    best_nb, best_alpha = max(cnb_scores, key=cnb_scores.get)

    # Keep the better variant by validation F1 (unless the caller fixes it)
    if variant is None:
        variant = ("CategoricalNB" if max(cnb_scores.values()) >= max(gnb_scores.values())
                   else "GaussianNB")
    if variant == "CategoricalNB":
        final = clone(cnb).set_params(preprocessor__num__n_bins=best_nb,
                                      classifier__alpha=best_alpha)
    else:
        final = clone(gnb).set_params(classifier__var_smoothing=best_vs)
    final.fit(Xtr, y_train)

    # Threshold by validation F1
    val_prob = final.predict_proba(Xva)[:, 1]
    thr = max(THRESHOLDS, key=lambda t: f1_score(y_val, (val_prob >= t).astype(int)))
    val_auc = roc_auc_score(y_val, val_prob)

    # Evaluate once on test
    test_prob = final.predict_proba(Xte)[:, 1]
    pred = (test_prob >= thr).astype(int)
    result = {
        "Variant": variant,
        "Params": (f"n_bins={best_nb}, alpha={best_alpha}" if variant == "CategoricalNB"
                   else f"var_smoothing={best_vs}"),
        "Threshold": thr,
        "Val ROC-AUC": val_auc,
        "Test Accuracy": accuracy_score(y_test, pred),
        "Test Precision": precision_score(y_test, pred),
        "Test Recall": recall_score(y_test, pred),
        "Test F1": f1_score(y_test, pred),
        "Test ROC-AUC": roc_auc_score(y_test, test_prob),
        "n_features": Xtr.shape[1],
    }
    return result, test_prob


In [17]:
# Run A (with history) and B (no history)
# The NB variant is fixed to CategoricalNB (the main NB model) so that A and B differ only in features
res_A, prob_A = run_nb_experiment([], variant="CategoricalNB")
res_B, prob_B = run_nb_experiment(history_features, variant="CategoricalNB")
rq1 = pd.DataFrame({"A: with history": res_A, "B: no history": res_B}).T
rq1_num = rq1[["Val ROC-AUC", "Test Accuracy", "Test Precision",
               "Test Recall", "Test F1", "Test ROC-AUC"]].astype(float).round(4)
print(rq1[["Variant", "Params", "Threshold", "n_features"]].to_string())
print()
print(rq1_num.to_string())
print("\nA - B (test):")
print((rq1_num.loc["A: with history"] - rq1_num.loc["B: no history"]).round(4).to_string())


                       Variant                 Params Threshold n_features
A: with history  CategoricalNB  n_bins=20, alpha=0.01       0.3         24
B: no history    CategoricalNB     n_bins=5, alpha=10      0.35         21

                 Val ROC-AUC  Test Accuracy  Test Precision  Test Recall  Test F1  Test ROC-AUC
A: with history       0.8270         0.7403          0.6222       0.6835   0.6514        0.8146
B: no history         0.8209         0.7481          0.6411       0.6596   0.6502        0.8084

A - B (test):
Val ROC-AUC       0.0061
Test Accuracy    -0.0078
Test Precision   -0.0189
Test Recall       0.0239
Test F1           0.0012
Test ROC-AUC      0.0062


In [18]:
# RQ1: is the A - B difference larger than test-sample noise? (paired bootstrap)
# Resample the test bookings with replacement (same rows for A and B), recompute the metric
# for both, and take the 2.5th / 97.5th percentiles of the difference as a 95% CI.
# Only test-sample variability is captured; hyperparameters and the data split are held fixed.
def paired_bootstrap_diff(y, s_a, s_b, metric, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)
    y, s_a, s_b = np.asarray(y), np.asarray(s_a), np.asarray(s_b)
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        i = rng.integers(0, len(y), len(y))
        diffs[b] = metric(y[i], s_a[i]) - metric(y[i], s_b[i])
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return metric(y, s_a) - metric(y, s_b), lo, hi

# ROC-AUC uses the probabilities; F1 uses each run's own validation-selected threshold
pred_A = (prob_A >= res_A["Threshold"]).astype(int)
pred_B = (prob_B >= res_B["Threshold"]).astype(int)

rows = []
for name, s_a, s_b, metric in [
    ("ROC-AUC", prob_A, prob_B, roc_auc_score),
    ("F1", pred_A, pred_B, f1_score),
]:
    diff, lo, hi = paired_bootstrap_diff(y_test, s_a, s_b, metric)
    rows.append({"Metric": name, "A - B": diff, "CI 2.5%": lo, "CI 97.5%": hi,
                 "CI excludes 0": bool(lo > 0 or hi < 0)})
rq1_bootstrap = pd.DataFrame(rows)
print(rq1_bootstrap.round(4).to_string(index=False))

# Where does the gain come from? History information exists for only a minority of bookings
has_history = ((X_test["previous_cancellations"] > 0)
               | (X_test["previous_bookings_not_canceled"] > 0)
               | (X_test["is_repeated_guest"] > 0)).values
print(f"\nTest bookings with any customer history: {has_history.mean():.1%} "
      f"(cancel rate {y_test[has_history].mean():.1%} vs {y_test[~has_history].mean():.1%} without)")
for label, mask in [("with history", has_history), ("without history", ~has_history)]:
    print(f"ROC-AUC on bookings {label:16s}: A = {roc_auc_score(y_test[mask], prob_A[mask]):.4f}, "
          f"B = {roc_auc_score(y_test[mask], prob_B[mask]):.4f}")

rq1.to_csv(output_dir / "naive_bayes_RQ1_results.csv")
rq1_bootstrap.to_csv(output_dir / "naive_bayes_RQ1_bootstrap.csv", index=False)

 Metric  A - B  CI 2.5%  CI 97.5%  CI excludes 0
ROC-AUC 0.0062    0.004    0.0084           True
     F1 0.0012   -0.004    0.0062          False

Test bookings with any customer history: 8.2% (cancel rate 56.3% vs 33.6% without)
ROC-AUC on bookings with history    : A = 0.9797, B = 0.9549
ROC-AUC on bookings without history : A = 0.7850, B = 0.7848


In [19]:
# RQ2: train on one hotel, test on both (hotel column dropped as it is constant)

hotels = ["Resort Hotel", "City Hotel"]

def split_by_hotel(df, hotel):
    d = df[df["hotel"] == hotel]
    return d.drop(columns=[target, "hotel"]), d[target]

def fit_nb_on_hotel(source):
    Xtr, ytr = split_by_hotel(train, source)
    Xva, yva = split_by_hotel(val, source)

    cat = [c for c in categorical_features if c != "hotel"]
    num = [c for c in Xtr.columns if c not in cat]

    # GaussianNB: tune var_smoothing
    gnb = Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", StandardScaler(), num),
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat)
        ])),
        ("classifier", GaussianNB())
    ])
    gnb_scores = {}
    for vs in [1e-9, 1e-7, 1e-5, 1e-3, 1e-1]:
        m = clone(gnb).set_params(classifier__var_smoothing=vs).fit(Xtr, ytr)
        gnb_scores[vs] = f1_score(yva, m.predict(Xva))
    best_vs = max(gnb_scores, key=gnb_scores.get)

    # CategoricalNB: tune n_bins and alpha
    cnb = Pipeline([
        ("preprocessor", ColumnTransformer([
            ("num", KBinsDiscretizer(n_bins=10, encode="ordinal", strategy="quantile",
                                     quantile_method="averaged_inverted_cdf"), num),
            ("cat", Pipeline([
                ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value",
                                           unknown_value=-1)),
                ("shift", FunctionTransformer(lambda a: a + 1))
            ]), cat)
        ])),
        ("classifier", CategoricalNB())
    ])
    cnb_scores = {}
    for n_bins in [5, 10, 20]:
        for alpha in [0.01, 0.1, 1, 10]:
            m = clone(cnb).set_params(preprocessor__num__n_bins=n_bins,
                                      classifier__alpha=alpha).fit(Xtr, ytr)
            cnb_scores[(n_bins, alpha)] = f1_score(yva, m.predict(Xva))
    best_nb, best_alpha = max(cnb_scores, key=cnb_scores.get)

    # Keep the better variant by validation F1
    if max(cnb_scores.values()) >= max(gnb_scores.values()):
        variant = "CategoricalNB"
        params = f"n_bins={best_nb}, alpha={best_alpha}"
        model = clone(cnb).set_params(preprocessor__num__n_bins=best_nb,
                                      classifier__alpha=best_alpha)
    else:
        variant = "GaussianNB"
        params = f"var_smoothing={best_vs}"
        model = clone(gnb).set_params(classifier__var_smoothing=best_vs)
    model.fit(Xtr, ytr)

    # Threshold by validation F1 on the source hotel
    val_prob = model.predict_proba(Xva)[:, 1]
    thr = max(THRESHOLDS, key=lambda t: f1_score(yva, (val_prob >= t).astype(int)))
    return model, thr, variant, params, len(Xtr)

rq2_rows = []
rq2_info = []
for source in hotels:
    model, thr, variant, params, n_train = fit_nb_on_hotel(source)
    rq2_info.append({"Train hotel": source, "n_train": n_train, "Variant": variant,
                     "Params": params, "Threshold": thr})
    for target_hotel in hotels:
        Xte, yte = split_by_hotel(test, target_hotel)
        prob = model.predict_proba(Xte)[:, 1]
        pred = (prob >= thr).astype(int)
        rq2_rows.append({
            "Train": source,
            "Test": target_hotel,
            "Type": "in-hotel" if source == target_hotel else "transfer",
            "n_test": len(yte),
            "Cancel rate": yte.mean(),
            "Accuracy": accuracy_score(yte, pred),
            "Precision": precision_score(yte, pred),
            "Recall": recall_score(yte, pred),
            "F1": f1_score(yte, pred),
            "ROC-AUC": roc_auc_score(yte, prob),
        })

print(pd.DataFrame(rq2_info).to_string(index=False))
print()
rq2 = pd.DataFrame(rq2_rows)
print(rq2.round(4).to_string(index=False))


 Train hotel  n_train       Variant                Params  Threshold
Resort Hotel    28254 CategoricalNB n_bins=20, alpha=0.01        0.4
  City Hotel    55561 CategoricalNB   n_bins=20, alpha=10        0.3

       Train         Test     Type  n_test  Cancel rate  Accuracy  Precision  Recall     F1  ROC-AUC
Resort Hotel Resort Hotel in-hotel    5996       0.2809    0.7702     0.5745  0.7007 0.6314   0.8210
Resort Hotel   City Hotel transfer   11342       0.3942    0.7351     0.6999  0.5739 0.6307   0.7806
  City Hotel Resort Hotel transfer    5996       0.2809    0.7158     0.4955  0.6526 0.5633   0.7828
  City Hotel   City Hotel in-hotel   11342       0.3942    0.7265     0.6629  0.6229 0.6423   0.7967
